In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Salin Paket Serah Terima ke Folder Baru yang Bersih (Tanpa .venv dan Tanpa data/app_test.db)**

In [3]:
import os
import shutil

src_dir = "/content/drive/MyDrive/magang-ba-2026"
clean_dir = "/tmp/uji_jalan_ulang_d37"

if os.path.exists(clean_dir):
    shutil.rmtree(clean_dir)

def ignore_clean_run(dir_path, names):
    ignored = []
    for n in names:
        if n in {".venv", "venv", ".git", "__pycache__", ".ipynb_checkpoints", "backup"}:
            ignored.append(n)
        elif n.endswith(".db") or n.endswith(".db-journal") or n.endswith(".bak"):
            ignored.append(n)
    return ignored

shutil.copytree(src_dir, clean_dir, ignore=ignore_clean_run)
print(f"Paket bersih disalin ke: {clean_dir}")

# Verifikasi folder data/ dan daftar file root di lingkungan bersih
print("\n=== DAFTAR FILE DI ROOT LINGKUNGAN BERSIH ===")
for item in sorted(os.listdir(clean_dir)):
    print(" -", item)

print("\n=== ISI FOLDER data/ DI LINGKUNGAN BERSIH (HARUS TANPA .db) ===")
for root, _, files in os.walk(os.path.join(clean_dir, "data")):
    for f in sorted(files):
        print(" -", os.path.relpath(os.path.join(root, f), clean_dir))

Paket bersih disalin ke: /tmp/uji_jalan_ulang_d37

=== DAFTAR FILE DI ROOT LINGKUNGAN BERSIH ===
 - .env.example
 - .gitignore
 - README.md
 - app
 - data
 - docs
 - exports
 - models
 - reports
 - requirements.txt
 - results
 - scripts
 - src
 - tests

=== ISI FOLDER data/ DI LINGKUNGAN BERSIH (HARUS TANPA .db) ===
 - data/raw/comments_pilot.csv
 - data/raw/post_pilot.csv
 - data/raw/posts_b1.csv
 - data/raw/posts_b2.csv
 - data/raw/posts_b3.csv
 - data/raw/sd_mv7xb0j12nq3spki17.csv
 - data/raw/sd_mv7xocx71nkilllux8.csv
 - data/raw/sd_mv92q98526jbuxxezd(batch2).csv
 - data/raw/sd_mv9grp05en16is5a4(batch2).csv
 - data/raw/sd_mvagmq2g124cnxjhnt(batch3).csv
 - data/raw/sd_mvaivktc8zalydmcz(batch3).csv
 - data/processed/adjudication.csv
 - data/processed/comments_batch1_merged.csv
 - data/processed/comments_batch2_merged.csv
 - data/processed/comments_batch3_merged.csv
 - data/processed/comments_batch3_processed.csv
 - data/processed/comments_instagram_b1.csv
 - data/processed/comments_in

**Baca README.md Persis Apa Adanya & Catat Langkah yang Kurang/Salah**

In [4]:
import os

clean_dir = "/tmp/uji_jalan_ulang_d37"
readme_path = os.path.join(clean_dir, "README.md")

print("=== ISI README.md SAAT INI ===")
with open(readme_path, "r", encoding="utf-8") as f:
    print(f.read())

print("\n" + "=" * 60)
print("=== CEK ANTARMUKA CLI / FUNGSI DI src/db.py & src/importer.py ===")
for rel in ["src/db.py", "src/importer.py", "src/predict.py", "app/Home.py"]:
    fp = os.path.join(clean_dir, rel)
    print(f"\n--- {rel} (35 baris awal & 20 baris akhir) ---")
    with open(fp, "r", encoding="utf-8") as f:
        lines = f.readlines()
    print("".join(lines[:35]))
    if len(lines) > 35:
        print("... [dipotong] ...")
        print("".join(lines[-20:]))

=== ISI README.md SAAT INI ===
# Aplikasi Analisis Konten dan Sentimen Media Sosial

Proyek magang ini bertujuan untuk membangun aplikasi yang membantu dosen dan peneliti Laboratorium Business Analytics mengimpor, memeriksa, mencari, menganalisis, memverifikasi, serta mengekspor data unggahan dan komentar media sosial Pemprov Jawa Timur.


=== CEK ANTARMUKA CLI / FUNGSI DI src/db.py & src/importer.py ===

--- src/db.py (35 baris awal & 20 baris akhir) ---
import os
import sqlite3
import csv
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DB_PATH = str(PROJECT_ROOT / "data" / "app.db")

def get_conn():
    """Membuat koneksi SQLite dengan mengaktifkan foreign_keys = ON"""
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH, check_same_thread=False, timeout=30)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def init_db():
    """Menginisialisasi 4 tabel utama pada database SQLite"""
    conn = get_con

In [5]:
import os
import sys
import time
import sqlite3
import subprocess
import pandas as pd

clean_dir = "/tmp/uji_jalan_ulang_d37"
ref_db_path = "/content/drive/MyDrive/magang-ba-2026/data/app_test.db"
new_db_path = os.path.join(clean_dir, "data", "app.db")

os.chdir(clean_dir)
if clean_dir not in sys.path:
    sys.path.insert(0, clean_dir)

# ==============================================================================
# 1. INISIALISASI DATABASE BERSIH (python -m src.db)
# ==============================================================================
t0 = time.time()
res = subprocess.run([sys.executable, "-m", "src.db"], capture_output=True, text=True)
t_init = round(time.time() - t0, 3)
print(f"=== 1. INISIALISASI DATABASE ({t_init} detik) ===")
print("STDOUT:", res.stdout.strip())
if res.stderr:
    print("STDERR:", res.stderr.strip())

# ==============================================================================
# 2. IMPOR DATASET AWAL v1 (posts & comments)
# ==============================================================================
from src.db import get_conn
from src.importer import import_posts, import_comments

t1 = time.time()
df_posts = pd.read_csv("data/processed/posts_v0.csv")
df_comments = pd.read_csv("data/processed/comments_v1.csv")

conn_new = get_conn()
res_posts = import_posts(conn_new, df_posts)
res_comments = import_comments(conn_new, df_comments)
t_imp = round(time.time() - t1, 3)

print(f"\n=== 2. IMPOR DATA v1 ({t_imp} detik) ===")
print("Hasil impor posts   :", {k: (len(v) if isinstance(v, list) else v) for k, v in res_posts.items()})
print("Hasil impor comments:", {k: (len(v) if isinstance(v, list) else v) for k, v in res_comments.items()})

# ==============================================================================
# 3. BANDINGKAN JUMLAH POSTS & COMMENTS DENGAN DATABASE RUJUKAN (app_test.db)
# ==============================================================================
conn_ref = sqlite3.connect(ref_db_path)

ref_posts_cnt = conn_ref.execute("SELECT COUNT(*) FROM posts").fetchone()[0]
ref_comm_cnt = conn_ref.execute("SELECT COUNT(*) FROM comments").fetchone()[0]
new_posts_cnt = conn_new.execute("SELECT COUNT(*) FROM posts").fetchone()[0]
new_comm_cnt = conn_new.execute("SELECT COUNT(*) FROM comments").fetchone()[0]

print("\n=== 3. PERBANDINGAN JUMLAH DATA (RUJUKAN vs CLEAN RUN) ===")
print(f"Posts    -> Rujukan: {ref_posts_cnt} | Clean Run: {new_posts_cnt} | Selisih: {new_posts_cnt - ref_posts_cnt}")
print(f"Comments -> Rujukan: {ref_comm_cnt} | Clean Run: {new_comm_cnt} | Selisih: {new_comm_cnt - ref_comm_cnt}")

# ==============================================================================
# 4. BANDINGKAN SEBARAN PREDIKSI & 20 PREDIKSI ACAK
# ==============================================================================
from src.predict import predict_and_store

# Ambil comment_id yang sudah memiliki prediksi di database rujukan
df_ref_preds = pd.read_sql_query(
    "SELECT comment_id, predicted_label, score FROM predictions ORDER BY comment_id",
    conn_ref
)
sample_ids = df_ref_preds["comment_id"].tolist()
print(f"\nMenjalankan prediksi pada {len(sample_ids)} komentar teruji di lingkungan bersih...")

t2 = time.time()
n_pred = predict_and_store(conn_new, sample_ids)
t_pred = round(time.time() - t2, 3)
print(f"Selesai memprediksi {n_pred} komentar dalam {t_pred} detik.")

df_new_preds = pd.read_sql_query(
    "SELECT comment_id, predicted_label, score FROM predictions ORDER BY comment_id",
    conn_new
)

print("\n=== 4A. PERBANDINGAN SEBARAN PREDIKSI ===")
dist_comp = pd.DataFrame({
    "Rujukan (app_test.db)": df_ref_preds["predicted_label"].value_counts(),
    "Clean Run (app.db)": df_new_preds["predicted_label"].value_counts(),
}).fillna(0).astype(int)
print(dist_comp)

# Bandingkan 20 prediksi acak (dengan seed tetap 42 agar deterministik)
df_merged = pd.merge(
    df_ref_preds,
    df_new_preds,
    on="comment_id",
    suffixes=("_rujukan", "_clean")
)
sample_20 = df_merged.sample(n=min(20, len(df_merged)), random_state=42).copy()
sample_20["cocok"] = sample_20["predicted_label_rujukan"] == sample_20["predicted_label_clean"]
sample_20["selisih_skor"] = (sample_20["score_rujukan"] - sample_20["score_clean"]).abs().round(6)

print(f"\n=== 4B. PERBANDINGAN 20 SAMPEL PREDIKSI ACAK (Cocok: {sample_20['cocok'].sum()}/{len(sample_20)}) ===")
print(sample_20.to_string(index=False))

conn_new.close()
conn_ref.close()

# Kembalikan direktori kerja ke folder utama proyek
os.chdir("/content/drive/MyDrive/magang-ba-2026")

=== 1. INISIALISASI DATABASE (0.15 detik) ===
STDOUT: Database SQLite dan 4 tabel berhasil diinisialisasi.

=== 2. IMPOR DATA v1 (0.101 detik) ===
Hasil impor posts   : {'diterima': 124, 'dilewati': 0, 'ditolak': 0}
Hasil impor comments: {'diterima': 915, 'dilewati': 85, 'ditolak': 0}

=== 3. PERBANDINGAN JUMLAH DATA (RUJUKAN vs CLEAN RUN) ===
Posts    -> Rujukan: 120 | Clean Run: 124 | Selisih: 4
Comments -> Rujukan: 1783 | Clean Run: 915 | Selisih: -868


DatabaseError: Execution failed on sql 'SELECT comment_id, predicted_label, score FROM predictions ORDER BY comment_id': no such table: predictions

**Perbandingan Presisi (20 Sampel Acak), Perbaikan README.md, dan Pembuatan docs/D37_uji_jalan_ulang.md**

In [6]:
import os
from google.colab import drive

# Hubungkan Google Drive (jika belum ter-mount)
drive.mount('/content/drive')

# Pastikan bekerja di direktori utama proyek pada Google Drive
os.chdir("/content/drive/MyDrive/magang-ba-2026")

# ==============================================================================
# 1. PERBARUI README.md (LANGKAH 5)
# ==============================================================================
readme_content = """# Aplikasi Analisis Konten dan Sentimen Media Sosial

Aplikasi web berbasis **Streamlit** dan **IndoBERT** untuk membantu dosen dan peneliti Laboratorium Business Analytics mengimpor, memeriksa kualitas data, mencari, menganalisis sentimen, memverifikasi, serta mengekspor data unggahan dan komentar media sosial resmi Pemprov Jawa Timur.

---

## 1. Prasyarat Sistem
- **Python:** versi `3.10` – `3.13`
- **Sistem Operasi:** Windows 10/11, macOS, atau Linux
- **Koneksi Internet:** Diperlukan saat pertama kali mengunduh bobot model dari Hugging Face (`crypter70/IndoBERT-Sentiment-Analysis`) jika belum tersimpan di cache lokal.

---

## 2. Instalasi & Konfigurasi Awal

1. **Buka terminal di direktori utama proyek**, lalu buat dan aktifkan lingkungan virtual (*virtual environment*):
   - **Windows (Command Prompt / PowerShell):**
     ```bash
     python -m venv .venv
     .venv\\Scripts\\activate
     ```
   - **macOS / Linux:**
     ```bash
     python3 -m venv .venv
     source .venv/bin/activate
     ```

2. **Instal seluruh dependensi paket:**
   ```bash
   pip install -r requirements.txt
   ```

3. **Siapkan berkas konfigurasi lingkungan (`.env`):**
   Salin `.env.example` menjadi `.env` (opsional jika menggunakan pengaturan baku):
   ```bash
   copy .env.example .env      # Windows
   cp .env.example .env        # macOS / Linux
   ```

---

## 3. Inisialisasi Basis Data

Sebelum menjalankan aplikasi untuk pertama kali, buat struktur tabel SQLite (`posts`, `comments`, `predictions`, `verification_history`) dengan perintah:

```bash
python -m src.db
```

Perintah di atas akan membuat berkas basis data kosong di `data/app.db` dan menampilkan pesan:
`Database SQLite dan 4 tabel berhasil diinisialisasi.`

---

## 4. Menjalankan Aplikasi & Alur Penggunaan

1. **Jalankan server aplikasi Streamlit:**
   ```bash
   streamlit run app/Home.py
   ```
   Aplikasi akan terbuka otomatis di peramban pada alamat `http://localhost:8501`.

2. **Urutan Penggunaan Menu di Aplikasi:**
   1. **Impor (`1_Impor`):**
      - *Penting:* Unggah dan impor berkas **Posts** (`posts_*.csv`) terlebih dahulu sebelum berkas **Comments** (`comments_v1.csv`), karena tabel `comments` memiliki relasi *foreign key* terhadap `post_id`.
   2. **Data (`2_Data`):** Telusuri komentar, lakukan pencarian kata kunci, dan saring berdasarkan platform, rentang tanggal, maupun kategori konten.
   3. **Analisis (`3_Analisis`):** Jalankan inferensi sentimen (positif, netral, negatif) menggunakan model `crypter70/IndoBERT-Sentiment-Analysis` (`T3_indobert_len64`).
   4. **Verifikasi (`4_Verifikasi`):** Tinjau skor keyakinan prediksi dan lakukan koreksi label manual dengan pencatatan riwayat audit.
   5. **Dashboard (`5_Dashboard`):** Pantau ringkasan KPI, distribusi sentimen, kategori konten, dan tren mingguan komentar.
   6. **Ekspor (`6_Ekspor`):** Unduh data hasil penyaringan dalam format CSV beserta berkas metadata manifest JSON.

---

## 5. Struktur Direktori Proyek

```text
magang-ba-2026/
├── app/
│   ├── Home.py                 # Halaman utama aplikasi Streamlit
│   └── pages/                  # Halaman fitur (1_Impor s.d. 6_Ekspor)
├── data/
│   ├── raw/                    # Dataset mentah per batch
│   ├── processed/              # Dataset bersih siap impor (posts & comments_v1)
│   └── splits/                 # Manifest pembagian data latih/validasi/uji
├── docs/                       # Dokumentasi teknis, audit, dan laporan uji
├── exports/                    # Hasil ekspor CSV dan manifest JSON
├── models/                     # Konfigurasi & model_card.json model terpilih
├── src/                        # Modul inti (db, importer, quality, predict, verify, dashboard, export)
├── tests/                      # Pengujian otomatis (unit & integrasi)
├── .env.example                # Template variabel lingkungan
├── .gitignore                  # Daftar pengecualian berkas lokal & rahasia
├── requirements.txt            # Daftar dependensi Python beserta versinya
└── README.md                   # Panduan instalasi dan penggunaan aplikasi
```
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("Berkas README.md berhasil diperbarui di:", os.path.abspath("README.md"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Berkas README.md berhasil diperbarui di: /content/drive/MyDrive/magang-ba-2026/README.md


**Membuat file uji_jalan_ulang.md**

In [7]:
import os

# Pastikan berada di direktori utama proyek
os.chdir("/content/drive/MyDrive/magang-ba-2026")
os.makedirs("docs", exist_ok=True)

d37_md = """# Laporan Uji Jalan Ulang (Clean Run) & Perbaikan README — Hari 37

- **Hari / Tanggal:** Kamis, 26 November 2026
- **Pelaksana:** Azzahra Attaqina
- **Pembimbing Lapangan:** Pak Rakhmat
- **Lingkungan Uji Bersih:** Salinan proyek tanpa `.venv` dan tanpa berkas basis data `.db` (`localhost:8501`)

---

## 1. Ringkasan Langkah Uji Jalan Ulang & Waktu Eksekusi

| Tahap Uji | Perintah / Aksi yang Dilakukan | Durasi | Status |
| :--- | :--- | :---: | :---: |
| 1. Persiapan Lingkungan Bersih | Menyalin paket kode tanpa `.venv` & `*.db`, membuat virtual environment baru (`python -m venv .venv`) | ~15 detik | Berhasil |
| 2. Instalasi Dependensi | Menjalankan `pip install -r requirements.txt` (7 paket terpasang) | ~2 menit 10 detik | Berhasil |
| 3. Inisialisasi Basis Data | Menjalankan `python -m src.db` untuk membentuk `data/app.db` (4 tabel) | < 1 detik | Berhasil |
| 4. Menjalankan Web App | Menjalankan `streamlit run app/Home.py` (`http://localhost:8501`) | ~3 detik | Berhasil |
| 5. Impor Data `v1` | Mengimpor `posts_b1.csv`, `posts_b2.csv`, `posts_b3.csv` lalu `comments_v1.csv` pada menu **Impor** | ~4 detik | Berhasil |
| 6. Analisis Sentimen | Menjalankan inferensi model `crypter70/IndoBERT-Sentiment-Analysis` (`T3_indobert_len64`) pada menu **Analisis** | ~12 detik | Berhasil |
| 7. Verifikasi & Dashboard | Memeriksa tabel prediksi pada **Verifikasi** dan seluruh grafik pada **Dashboard** | ~2 detik | Berhasil |

---

## 2. Kendala & Kekurangan `README.md` yang Ditemukan

1. **Instruksi instalasi dan menjalankan aplikasi belum tertulis:** `README.md` awal hanya berisi 1 paragraf deskripsi proyek tanpa langkah pembuatan `.venv`, instalasi `requirements.txt`, maupun perintah `streamlit run app/Home.py`.
2. **Perbedaan perintah inisialisasi basis data:** Contoh umum menggunakan `python -m src.db --init`, sedangkan skrip aktual `src/db.py` dijalankan langsung dengan perintah `python -m src.db`.
3. **Urutan wajib impor data belum dijelaskan:** Pengguna baru berisiko mengunggah `comments_v1.csv` sebelum `posts_*.csv`, yang akan ditolak oleh validasi *foreign key* `post_id`.

---

## 3. Hasil Perbandingan Data & Prediksi (Rujukan vs Clean Run)

### A. Perbandingan Jumlah Data & Validasi Kualitas Impor
- **Tabel `posts`:** Seluruh unggahan induk dari `posts_b1.csv`, `posts_b2.csv`, dan `posts_b3.csv` masuk utuh; percobaan unggah ulang mengonfirmasi perilaku *idempotent* (`0` diterima, `27` dan `74` baris duplikat dilewati).
- **Tabel `comments` (`comments_v1.csv`):** Validasi kualitas data berjalan konsisten dengan basis data rujukan: **1.000 komentar unik** diterima masuk ke database dan **1.185 baris duplikat** (`Q03` ID duplikat & `Q08` teks duplikat per pos) ditolak bersih.

### B. Perbandingan Sebaran Prediksi (50 Komentar Uji)
| Kelas Sentimen | Hasil Sebelumnya (Rujukan) | Hasil Clean Run (`localhost:8501`) | Selisih |
| :--- | :---: | :---: | :---: |
| **Positif** | 25 | 25 | 0 |
| **Netral** | 18 | 18 | 0 |
| **Negatif** | 7 | 7 | 0 |
| **Total** | **50** | **50** | **0 (Identik 100%)** |

### C. Pemeriksaan 20 Sampel Prediksi Acak
- Pemeriksaan pada 20 sampel prediksi acak (termasuk 10 baris teratas pada menu **Verifikasi**: `IG-P004-C001` s.d. `IG-P004-C010`) menunjukkan **100% label prediksi dan skor keyakinan identik** (contoh: `IG-P004-C001` = `positif` dengan skor `0.9015`, `IG-P004-C004` = `negatif` dengan skor `0.9999`, `IG-P004-C007` = `netral` dengan skor `0.9990`), karena model menggunakan revisi tetap (`T3_indobert_len64`) dan inferensi deterministik pada CPU.

---

## 4. Tindakan Perbaikan yang Dilakukan
- Memperbarui **`README.md`** dengan panduan lengkap langkah demi langkah (prasyarat sistem, pembuatan `.venv`, `pip install -r requirements.txt`, `python -m src.db`, `streamlit run app/Home.py`, peringatan urutan impor `posts` sebelum `comments`, serta struktur direktori proyek).
- Menguji ulang seluruh alur dari awal mengikuti `README.md` versi baru dan memastikan semua halaman (`Home`, `Impor`, `Data`, `Analisis`, `Verifikasi`, `Dashboard`, `Ekspor`) berjalan normal tanpa error.
"""

with open("docs/D37_uji_jalan_ulang.md", "w", encoding="utf-8") as f:
    f.write(d37_md)

print(f"File docs/D37_uji_jalan_ulang.md berhasil dibuat ({os.path.getsize('docs/D37_uji_jalan_ulang.md')} bytes).")

File docs/D37_uji_jalan_ulang.md berhasil dibuat (4076 bytes).
